# LightGBM Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Archivos necesarios en la misma carpeta:**
- `lgbm_cloud_best.txt` — modelo entrenado
- `lgbm_cloud_metadata.json` — metadata

In [1]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

print(f'LightGBM version: {lgb.__version__}')
print('Imports OK')

LightGBM version: 4.5.0
Imports OK


In [2]:
MODEL_DIR = Path('.')
DATA_DIR  = Path('../../data/processed')

model_file = MODEL_DIR / 'lgbm_cloud_best.txt'
meta_file  = MODEL_DIR / 'lgbm_cloud_metadata.json'

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
for f in [model_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle'
    print(f'  {f.name} : {status}')

MODEL_DIR : C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\cloud\lightgbm
  lgbm_cloud_best.txt : ✓ encontrado
  lgbm_cloud_metadata.json : ✓ encontrado


In [3]:
with open(meta_file, 'r') as f:
    meta = json.load(f)

print('Metadata del modelo:')
print(f'  Train AUC       : {meta["train_auc"]}')
print(f'  CV AUC (Optuna) : {meta["best_cv_auc"]}')
print(f'  n_rounds        : {meta["best_n_rounds"]}')
print(f'  n_features      : {len(meta["feature_cols"])}')
print(f'  GPU usado       : {meta["use_gpu"]}')
print(f'  LightGBM ver    : {meta["lgbm_version"]}')
print(f'  Timestamp       : {meta["timestamp"]}')

Metadata del modelo:
  Train AUC       : 0.8099
  CV AUC (Optuna) : 0.7507756653988509
  n_rounds        : 115
  n_features      : 30
  GPU usado       : True
  LightGBM ver    : 4.6.0
  Timestamp       : 2026-02-26T01:27:51.878578


In [4]:
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
model = lgb.Booster(model_file=str(model_file))
print('Modelo cargado ✓')

Cargando modelo: lgbm_cloud_best.txt  (0.32 MB)...
[LightGBM] [Warning] Ignoring unrecognized parameter 'bagging_by_query' found in model string.
Modelo cargado ✓


In [5]:
print('Cargando features_test.parquet...')
df_test      = pd.read_parquet(DATA_DIR / 'features_test.parquet')
sk_ids_test  = df_test['SK_ID_CURR'].values
feature_cols = meta['feature_cols']

# Encodear categóricas si existen
cat_cols = [c for c in feature_cols if df_test[c].dtype == 'object']
if cat_cols:
    for col in cat_cols:
        cats    = df_test[col].dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df_test[col] = df_test[col].map(mapping)

X_test = df_test[feature_cols].values
print(f'  Test shape  : {X_test.shape}')
print(f'  Features    : {len(feature_cols)}')

Cargando features_test.parquet...


  Test shape  : (48744, 30)
  Features    : 30


In [6]:
print('Generando predicciones...')
y_test_prob = model.predict(X_test)

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

Generando predicciones...
  Predicciones generadas : 48,744
  Score mínimo           : 0.01000
  Score máximo           : 0.96923
  Score medio            : 0.38287
  % predicho como default: 30.48%


In [7]:
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / 'submission_cloud_lgbm.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

Submission guardado: ..\..\data\processed\submission_cloud_lgbm.csv
Shape: (48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.341280
1,100005,0.689718
2,100013,0.198018
3,100028,0.207182
4,100038,0.650436


## Kaggle Submission — AUC Test (Public Leaderboard)

> **Límite**: 5 submissions/día.

In [8]:
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'
_msg = (f"LightGBM Cloud | "
        f"CV AUC={meta['best_cv_auc']:.5f} | "
        f"train AUC={meta['train_auc']:.5f} | "
        f"rounds={meta['best_n_rounds']}")

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(out_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 55)
print(f'RESULTADO KAGGLE — LIGHTGBM CLOUD')
print(f'=' * 55)
print(f'  AUC test Public  LB  : {public_auc}')
print(f'  AUC test Private LB  : {private_auc}')
print(f'  CV AUC (entrenamiento): {meta["best_cv_auc"]:.5f}')
if public_auc:
    print(f'  Gap CV - Public LB   : {abs(meta["best_cv_auc"] - public_auc):.5f}')
print(f'=' * 55)

Enviando: LightGBM Cloud | CV AUC=0.75078 | train AUC=0.80990 | rounds=115


100%|██████████| 1.27M/1.27M [00:01<00:00, 810kB/s] 


Esperando scoring (poll 10 s / máx 5 min)...
  [  0s] status='SubmissionStatus.PENDING'  public_score=''
  [ 10s] status='SubmissionStatus.COMPLETE'  public_score='0.77475'

RESULTADO KAGGLE — LIGHTGBM CLOUD
  AUC test Public  LB  : 0.77475
  AUC test Private LB  : 0.76734
  CV AUC (entrenamiento): 0.75078
  Gap CV - Public LB   : 0.02397
